<a href="https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/Copy_of_w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashir3585/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
## My Lane: Refresh / Content Opportunity Scoring

I'm choosing Lane 2 because it's a direct continuation of what I already built in Notebooks 01–03: a hand-written rule that scored pages by staleness and visibility, a readable decision tree that beat it, and a full pipeline on the 79M-row warehouse that engineered momentum + query-mix features and validated with client-grouped splits. This lane lets me go deeper into the same problem — ranking pages for review — rather than starting from scratch on a new question.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*
**Search question:** Which content pages should be reviewed first for a refresh, based on evidence that they're losing visibility while still holding real demand?

**Unit of analysis:** One row = one content page (content_hash_id), evaluated at a single point in time using its trailing performance window.

**Output:** A ranked list of pages, each with a score and a reason code (e.g. "stale_visible_page," "declining_with_demand," "low_ctr_visible_page").

**The action:** A content reviewer opens the top of the ranked list and decides, page by page, whether to refresh, protect, monitor, or leave it alone.

**Cost of a wrong call:** A false positive (flagged but not actually declining) wastes reviewer time — low cost. A false negative (a genuinely declining page never flagged) means lost visibility goes unnoticed and compounds — the more expensive mistake.

**Why data/ML can help:** With 500,000+ content items in the full release, no reviewer can check every page manually. A ranked, evidence-backed queue turns "check everything" into "check the top 50 that matter most."


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lakshya701/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())


Current directory: /content/flyrank-ml-internship


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

total_pages = len(df)
declining_rate = (df["trend_direction"].str.lower() == "down").mean()
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()

print(f"Total pages in starter slice: {total_pages:,}")
print(f"Declining rate: {declining_rate:.1%}")
print(f"Stale but visible pages: {stale_visible:,} ({stale_visible/total_pages:.1%})")

Total pages in starter slice: 30,000
Declining rate: 54.2%
Stale but visible pages: 17 (0.1%)


### Why This Lane Looks Worth 7 Weeks

Three numbers from the starter slice back this choice:

1. **54.2% of pages are currently declining** (trend_direction == "down") —
   more than half the dataset, so a ranking model has real, substantial signal
   to learn from, not a handful of edge cases.
2. **Only 17 pages (0.1%) are both stale (180+ days) and still pulling 500+
   impressions** — this is a small number in the 30k starter slice, but it's
   the exact shape of "worth reviewing" candidate the hand-rule in Notebook 02
   was built to catch. It also tells me my thresholds may need loosening once
   I move to the full 519,606-item warehouse, where this pattern should show
   up far more often.
3. **From Notebook 02, my own depth-3 decision tree already beat the hand rule
   at Precision@20 (0.700 vs 0.550) and Precision@50 (0.720 vs 0.600)** — real
   evidence, from my own run, that a learned model finds signal a fixed rule
   misses. On the full starter benchmark, random forest reaches Precision@50
   of 0.740 against a baseline of 0.240 — roughly 3x more of the top 50
   recommendations turn out correct.

The 54.2% decline rate shows this is a widespread, not rare, problem worth
solving. The small stale-visible count is a signal I need to test my
thresholds against the full warehouse rather than assume they hold at scale —
that's exactly the kind of check this lane is built to force me to make.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
**I can claim:** This is decision-support ranking, not a guarantee — it tells a reviewer where to look first. Results are observational, built on client-holdout validation on a small (~30k row) slice. A model beating a hand rule on Precision@K is real evidence it finds more signal, within this dataset's scope.

**I cannot claim:** That a refresh will cause recovery — that needs an actual experiment. Anything about Google's algorithm. That "declining" (trend_direction == "down") is the ideal target — it's a current-window bucket, not a future outcome; I plan to move toward a future-window label (prior 90 days → next 30 days decline) as I go deeper into this lane.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.